# Set-up

In [6]:
### Imports ###
import pandas as pd
import pickle
from skfda import FDataGrid
from datetime import date, datetime, timedelta
import numpy as np

from src.preprocessing import GMEPreprocessor, ExogPreprocessor
from src.curves import SupplyDemandTimeSeries
from src.forecasting import LassoVARX, SupplyDemandForecaster
from src.utils import fix_daylight_saving_time

In [11]:
### Parameters ###

# Input paths
bids_path = 'data/source/MGPDomandaOfferta/MGPDomandaOfferta.pkl'
coupling_path = 'data/source/MGPMarketCoupling/balance_coupling.csv'
prices_path = 'data/source/MGPPrezzi/mgp_prices.pkl'
exog_path = 'data/source/predictors.pkl'

# Processed paths
curves_path = 'data/processed/sdts.pkl'

# Output paths
pred_curves_path = 'data/output/sdts_pred_endog_l1_l2_l3_l7.pkl'
pred_prices_path = 'data/output/prices_pred_endog_l1_l2_l3_l7.pkl'

# Whether to re-preprocess curves (long!)
rerun_curves_preprocessing = False

# Model parameters

K_supply = 5
K_demand = 3

exog_variables = [
    'GFSo Solar ITA',
    'ECo Wind ITA',
    'Load_IT',
    # 'FR > IT',
    # 'IT > FR',
    # 'CH > IT',
    # 'IT > CH',
    # 'AT > IT',
    # 'IT > AT',
    # 'SI > IT',
    # 'IT > SI',
    # 'IT > ME',
    # 'ME > IT',
    # 'IT > GR',
    # 'GR > IT',
]

# Calibration window and testing period
calibration_window = timedelta(days=358)
test_start_date = date(2024, 1, 1)
test_end_date = date(2024, 12, 31)
print(f"Calibration_window: {calibration_window.days} days")
print(f"Testing period: {test_start_date} to {test_end_date}")

Calibration_window: 358 days
Testing period: 2024-01-01 to 2024-12-31


In [12]:
### Read data ###

bids = pd.read_pickle(bids_path)
coupling = pd.read_csv(coupling_path)
prices = pd.read_pickle(prices_path)
exog = pd.read_pickle(exog_path)

In [13]:
### Preprocessing ###

# Start and end datetimes
test_start = pd.Timestamp(test_start_date) # Time information automatically set at 00:00:00
test_end = pd.Timestamp(test_end_date) + timedelta(hours=23) # Time information set to 23:00:00
train_start = test_start - calibration_window
preprocess_start = train_start - timedelta(weeks=1) # We need one week of past data to compute the lags

# Curves
if rerun_curves_preprocessing:
    preprocessor = GMEPreprocessor()
    data_matrix_off, grid_points = preprocessor.get_curves_dataset(bids, type='OFF', balance_df=coupling)
    data_matrix_bid, _ = preprocessor.get_curves_dataset(bids, type='BID', balance_df=coupling)

    sd = SupplyDemandTimeSeries(
        FDataGrid(data_matrix_off, grid_points, sample_names=preprocessor.timestamps, extrapolation='bounds'),
        FDataGrid(data_matrix_bid, grid_points, sample_names=preprocessor.timestamps, extrapolation='bounds')
    )
    sd.to_pickle(curves_path)
else:
    with open(curves_path, 'rb') as file:
        sd = pickle.load(file)
sd = sd[preprocess_start:test_end]

# Exog
exogprep = ExogPreprocessor(
    start_date=preprocess_start.date(),
    end_date=test_end_date,
    exog_variables=exog_variables
)
exog = exogprep.preprocess_exog(exog)

# Prices
prices = fix_daylight_saving_time(prices)
prices_true = prices.loc[test_start:test_end, 'NAT']

In [21]:
### Forecasting ###

model = LassoVARX(
    lags_endogs=[1, 2, 3, 7],
    lags_exog=[0, 1, 7],
    ar_structure='full',
    var_structure='concurrent',
    exog_structure='full',
    calibration_window=calibration_window,
    criterion='cv',
    daytype_dummies=exogprep.dummy_columns,
    show_features=True,
    n_jobs=-1,
)

forecaster = SupplyDemandForecaster(model, exogprep, K_supply=K_supply, K_demand=K_demand)
sd_pred = forecaster.fit_forecast(sd, exog, test_start=date(2024, 1, 1), recalibration=None)
prices_pred = sd_pred.get_clearing_prices()
print("MAE: {:.2f}€/MWh".format((prices_true - prices_pred).abs().mean()))

2025-09-26 18:54:25,271 - INFO - 271 features for FPC1o hour 0: Index(['Load_IT_h0', 'RES_h0', 'is_Holiday', 'is_Monday', 'is_Saturday',
       'Load_IT_h0_L1', 'RES_h0_L1', 'Load_IT_h0_L7', 'RES_h0_L7',
       'Load_IT_h1',
       ...
       'FPC1b_h0_L3', 'FPC1b_h0_L7', 'FPC2b_h0_L1', 'FPC2b_h0_L2',
       'FPC2b_h0_L3', 'FPC2b_h0_L7', 'FPC3b_h0_L1', 'FPC3b_h0_L2',
       'FPC3b_h0_L3', 'FPC3b_h0_L7'],
      dtype='object', length=271)
2025-09-26 18:54:26,155 - INFO - Training period is from 2023-01-08 to 2023-12-31
2025-09-26 18:54:26,155 - INFO - Forecasting period is from 2024-01-01 to 2024-12-31


MAE: 9.53€/MWh


In [ ]:
### Evaluate and save ###

# sd_pred.to_pickle(pred_curves_path)
# prices_pred.to_pickle(pred_prices_path)
print("MAE: {:.2f}€/MWh".format((prices_true - prices_pred).abs().mean()))

MAE: 8.19€/MWh


<HR>

# Tests

In [16]:
sd_naive = sd[test_start - timedelta(days=7):].get_naive_forecast()
prices_naive = sd_naive.get_clearing_prices()
print("MAE: {:.2f}€/MWh".format((prices_true - prices_naive).abs().mean()))

MAE: 11.34€/MWh
